In [ ]:
MANIFEST_PATH = "Files/bronze/returns/landing/ingestion_date=2026-10-28/returns_anomaly_manifest.csv"

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.errors import AnalysisException

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

#accepted values: keep in step with scripts/generate_returns.py and docs/returns_source.md
REFUND_STATUSES = ["requested", "approved", "completed", "rejected", "cancelled"]
FINANCIAL_REFUND_STATUSES = ["approved", "completed"]
NON_CONSUMING_STATUSES = ["rejected", "cancelled"]
RETURN_REASONS = [
    "defective", "changed_mind", "not_as_described",
    "wrong_item", "damaged_in_transit", "late_delivery",
]
ITEM_CONDITIONS = ["new_unopened", "opened", "damaged", "defective"]
RETURN_CHANNELS = ["store", "online"]


def write_silver(df, table_name):
    (
        df.withColumn("_silver_processed_at_utc", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


def not_in(column_name, values):
    return ~F.coalesce(F.col(column_name).isin(values), F.lit(False))

In [ ]:
#type
bronze_df = spark.table("bronze.returns_raw")

typed_df = bronze_df.select(
    F.trim("return_id").alias("return_id"),
    F.trim("order_number").try_cast("int").alias("order_number"),
    F.trim("line_item").try_cast("int").alias("line_item"),
    F.trim("return_date_raw").try_cast("date").alias("return_date"),
    F.trim("returned_quantity_raw").try_cast("int").alias("returned_quantity"),
    F.lower(F.trim("return_reason_raw")).alias("return_reason"),
    F.lower(F.trim("item_condition_raw")).alias("item_condition"),
    F.lower(F.trim("return_channel_raw")).alias("return_channel"),
    F.trim("return_store_key").try_cast("int").alias("return_store_key"),
    F.lower(F.trim("refund_status_raw")).alias("refund_status"),
    F.trim("refund_date_raw").try_cast("date").alias("refund_date"),
    F.trim("refund_amount_raw").try_cast("decimal(18,2)").alias("refund_amount"),
    F.upper(F.trim("refund_currency_code_raw")).alias("refund_currency_code"),
    F.trim("status_updated_date_raw").try_cast("date").alias("status_updated_date"),
    "return_date_raw",
    "refund_date_raw",
    "refund_amount_raw",
    "source_batch_id",
    "generator_version",
    "generator_seed",
    "_batch_id",
    "_source_file",
    "_ingested_at_utc",
)

#the same return_id twice in one batch is a duplicate; across batches it is a newer version
in_batch_window = Window.partitionBy("_batch_id", "return_id").orderBy(
    F.col("status_updated_date").desc_nulls_last(),
    "return_date_raw", "refund_date_raw", "refund_amount_raw", "refund_status",
)
version_window = Window.partitionBy("return_id").orderBy(
    F.col("status_updated_date").desc_nulls_last(),
    F.col("_batch_id").desc(),
)

typed_df = typed_df.withColumn("_duplicate_rank", F.row_number().over(in_batch_window))
first_copies_df = typed_df.filter("_duplicate_rank = 1").withColumn(
    "_version_rank", F.row_number().over(version_window)
)

#older versions are superseded, not invalid: they stay in Bronze as history
current_df = first_copies_df.filter("_version_rank = 1").drop("_version_rank")
superseded_rows = first_copies_df.filter("_version_rank > 1").count()
duplicate_df = typed_df.filter("_duplicate_rank > 1")

In [ ]:
#enrich with the original sale; refunds use the sale's FX rates (ADR 003)
sales_df = spark.table("silver.sales_lines").select(
    "order_number",
    "line_item",
    "order_date",
    F.col("quantity").alias("sold_quantity"),
    F.col("currency_code").alias("sale_currency_code"),
    "unit_price_local",
    "order_rate_per_usd",
    "order_fx_rate_date",
    "eur_rate_per_usd",
)

#cumulative quantity counts every return that took units back, in return order
consumed_window = (
    Window.partitionBy("order_number", "line_item")
    .orderBy("return_date", "return_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

candidates_df = (
    current_df.unionByName(duplicate_df, allowMissingColumns=True)
    .join(sales_df, ["order_number", "line_item"], "left")
    .withColumn(
        "cumulative_returned_quantity",
        F.when(
            (F.col("_duplicate_rank") == 1) & not_in("refund_status", NON_CONSUMING_STATUSES),
            F.sum(
                F.when(
                    (F.col("_duplicate_rank") == 1) & not_in("refund_status", NON_CONSUMING_STATUSES),
                    F.col("returned_quantity"),
                ).otherwise(0)
            ).over(consumed_window),
        ),
    )
    .withColumn(
        "refundable_amount",
        (F.col("returned_quantity") * F.col("unit_price_local")).cast("decimal(18,2)"),
    )
)

In [ ]:
#data-quality rules: a row failing any rule is quarantined with every reason it failed
CHECKS = {
    "duplicate_return_id": F.col("_duplicate_rank") > 1,
    "invalid_return_id": F.col("return_id").isNull(),
    "orphan_order_line": F.col("sold_quantity").isNull(),
    "invalid_return_date": F.col("return_date").isNull(),
    "return_before_sale": F.col("return_date") < F.col("order_date"),
    "invalid_returned_quantity": F.col("returned_quantity").isNull() | (F.col("returned_quantity") <= 0),
    "quantity_exceeds_sold": F.col("cumulative_returned_quantity") > F.col("sold_quantity"),
    "unknown_return_reason": not_in("return_reason", RETURN_REASONS),
    "unknown_item_condition": not_in("item_condition", ITEM_CONDITIONS),
    "unknown_return_channel": not_in("return_channel", RETURN_CHANNELS),
    "unknown_refund_status": not_in("refund_status", REFUND_STATUSES),
    "inconsistent_return_store": (
        ((F.col("return_channel") == "store") & F.col("return_store_key").isNull())
        | ((F.col("return_channel") == "online") & F.col("return_store_key").isNotNull())
    ),
    "completed_without_refund_date": (F.col("refund_status") == "completed") & F.col("refund_date").isNull(),
    "unexpected_refund_date": (F.col("refund_status") != "completed") & F.col("refund_date").isNotNull(),
    "refund_before_return": F.col("refund_date") < F.col("return_date"),
    "missing_refund_amount": F.col("refund_status").isin(FINANCIAL_REFUND_STATUSES) & F.col("refund_amount").isNull(),
    "unexpected_refund_amount": not_in("refund_status", FINANCIAL_REFUND_STATUSES) & F.col("refund_amount").isNotNull(),
    "negative_refund_amount": F.col("refund_amount") < 0,
    "refund_exceeds_line_value": F.col("refund_amount") > F.col("refundable_amount"),
    "refund_currency_mismatch": F.col("refund_currency_code") != F.col("sale_currency_code"),
}

checked_df = candidates_df.withColumn(
    "failure_reasons",
    F.array_compact(F.array(*[
        F.when(F.coalesce(failed, F.lit(False)), F.lit(name)) for name, failed in CHECKS.items()
    ])),
)

returns_df = (
    checked_df.filter(F.size("failure_reasons") == 0)
    .withColumn("is_financial_refund", F.col("refund_status").isin(FINANCIAL_REFUND_STATUSES))
    .withColumn(
        "refund_amount_eur",
        F.round(
            F.col("refund_amount") / F.col("order_rate_per_usd") * F.col("eur_rate_per_usd"), 2
        ).cast("decimal(18,2)"),
    )
    .select(
        "return_id", "order_number", "line_item", "return_date", "returned_quantity",
        "return_reason", "item_condition", "return_channel", "return_store_key",
        "refund_status", "is_financial_refund", "refund_date",
        "refund_currency_code", "refund_amount",
        "order_rate_per_usd", "order_fx_rate_date", "eur_rate_per_usd", "refund_amount_eur",
        "status_updated_date",
        "source_batch_id", "generator_version", "generator_seed",
        "_batch_id", "_source_file", "_ingested_at_utc",
    )
)

write_silver(returns_df, "silver.returns")
write_silver(checked_df.filter(F.size("failure_reasons") > 0), "silver.returns_quarantine")

In [ ]:
#validate and reconcile with Bronze
bronze_rows = bronze_df.count()
silver_df = spark.table("silver.returns")
quarantine_df = spark.table("silver.returns_quarantine")
silver_rows = silver_df.count()
quarantine_rows = quarantine_df.count()

duplicate_ids = silver_df.groupBy("return_id").count().filter("count > 1").count()
over_returned_lines = (
    silver_df.filter(~F.col("refund_status").isin(NON_CONSUMING_STATUSES))
    .groupBy("order_number", "line_item").agg(F.sum("returned_quantity").alias("returned"))
    .join(spark.table("silver.sales_lines"), ["order_number", "line_item"])
    .filter(F.col("returned") > F.col("quantity"))
    .count()
)

print(f"Bronze rows: {bronze_rows:,}")
print(f"Silver rows (current versions): {silver_rows:,}")
print(f"Quarantined rows: {quarantine_rows:,}")
print(f"Superseded versions: {superseded_rows:,}")
print(f"Duplicate return_id in Silver: {duplicate_ids:,}")
print(f"Order lines returned beyond sold quantity: {over_returned_lines:,}")

assert silver_rows + quarantine_rows + superseded_rows == bronze_rows
assert duplicate_ids == 0
assert over_returned_lines == 0

display(
    quarantine_df.select("return_id", F.explode("failure_reasons").alias("failure_reason"))
    .groupBy("failure_reason").count().orderBy("failure_reason")
)

In [ ]:
#every injected anomaly must be quarantined for its own reason, and nothing else may be
try:
    manifest_df = spark.read.option("header", True).csv(MANIFEST_PATH)
except AnalysisException:
    manifest_df = None
    print(f"No manifest at {MANIFEST_PATH}; skipping the anomaly check")

if manifest_df is not None:
    caught_df = quarantine_df.select("return_id", F.explode("failure_reasons").alias("anomaly_type"))
    missed_df = manifest_df.join(caught_df, ["return_id", "anomaly_type"], "left_anti")
    unexpected_df = caught_df.select("return_id").distinct().join(manifest_df, "return_id", "left_anti")

    print(f"Manifest anomalies: {manifest_df.count():,}")
    print(f"Missed anomalies: {missed_df.count():,}")
    print(f"Quarantined records not in the manifest: {unexpected_df.count():,}")

    display(missed_df)
    display(unexpected_df)
    assert missed_df.count() == 0
    assert unexpected_df.count() == 0